<a href="https://colab.research.google.com/github/GrayCrossX/NTU/blob/main/Q3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from scipy.signal import butter, correlate, find_peaks, sosfiltfilt
from scipy.interpolate import interp1d
from sklearn.decomposition import PCA
from sklearn.covariance import MinCovDet

In [ ]:
def bandpass_preprocessing(x, fs):
  sos = butter(5, [0.5, 8.0], btype = "bandpass", fs = fs, output = "sos")
  return sosfiltfilt(sos, x)

def beat_segments(x, fs):
    y = bandpass_preprocessing(x, fs)
    peaks, props = find_peaks(y, distance = int(0.3*fs), prominence=np.std(y)/10)
    beats = []
    for p0, p1 in zip(peaks[:-1], peaks[1:]):
      beat = x[p0:p1]
      duration = len(beat)/fs
      if 0.3 <= duration <= 2.0:
        beats.append({"waveform": beat, "start": p0, "end": p1, "ibi": (p1 - p0)/fs})
    return beats

In [ ]:
def normalize_beat(beat):
    x = beat["waveform"]
    x = x - np.median(x)
    scale = np.percentile(x, 95) - np.percentile(x, 5)
    if scale < 1e-8:
      return None
    x = x/scale
    old_t = np.linspace(0, 1, len(x))
    new_t = np.linspace(0, 1, 128)
    T = np.interp(new_t, old_t, x)
    return T

In [ ]:
Z = np.stack([
    normalize_beat(b)
    for b in beats
    if normalize_beat(b) is not None
    ])
pca = PCA(n_components = 8, whiten = True)
E = pca.fit_transform(Z)
robust_cov = MinCovDet().fit(E)
mahalanobis = robust_cov.mahalanobis(E)

In [ ]:
def morphology_score(Z, i, radius=3):
  neighbours = [
      j for j in range(max(0, i - radius), min(len(Z), i + radius + 1))
      if j != i
      ]
  scores = []
  for j in neighbours:
    scores.append(np.corrcoef(Z[i], Z[j])[0, 1])
  M = np.median(scores)
  return M

def autocorrelation_structure(x):
  x = x - np.mean(x)
  r = correlate(x, x, mode = "full")
  r = r[len(r)//2:]
  r /= r[0] + 1e-8
  A = np.max(r[5:50])
  return A

def spectral_concentration(x):
  X = np.abs(np.fft.rfft(x))
  f = np.fft.rfftfreq(len(x))
  total = np.sum(X**2) + 1e-8
  low = np.sum(X[(f >= 0.005) & (f <= 0.150)]**2)
  S = low/total
  return S

In [ ]:
def physiological_augmentation(x):
  y = x.copy()
  y *= np.random.uniform(0.75, 1.25)
  y += np.random.uniform(-0.10, 0.10)
  y = small_time_warp(y)
  y += np.random.normal(0, 0.01, len(y))
  return y

for batch in beats:
  x1 = physiological_augmentation(batch)
  x2 = physiological_augmentation(batch)
  z1 = encoder(x1)
  z2 = encoder(x2)
  p1 = predictor(z1)
  p2 = predictor(z2)
  loss = (negative_cosine_similarity(p1, stop_gradient(z2)) + negative_cosine_similarity(p2, stop_gradient(z1))).mean()
  loss.backward()
  optimizer.step()

In [ ]:
embeddings = encoder(all_beats)
centre = np.median(embeddings, axis = 0)
distance = np.linalg.norm(embeddings - centre, axis = 1)

features = np.column_stack([
    embedding_distance,
    1 - morphology_agreement,
    1 - timing_consistency,
    1 - autocorrelation_score,
    1 - spectral_concentration,
])
ranks = robust_percentile_transform(features)
quality = 1.0 - np.mean(ranks, axis = 1)

In [ ]:
candidate_stable = (
    local_morphology_agreement
    &
    low_timing_variance
    &
    reasonable_signal_energy
)
seed_beats = longest_stable_runs(candidate_stable)

normal = seed_beats
for iteration in range(3):
  model.fit(normal)
  scores = model.distance(all_beats)
  normal = select_consensus_population(all_beats, scores, temporal_consistency = True)

In [ ]:
quality_label = np.where(
    1.00 >= quality >= 0.75, "good",
    0.75 > quality > 0.65, "possibly good",
    0.65 >= quality >= 0.45, "unknown",
    0.45 > quality > 0.25, "possibly bad",
    0.25 >= quality >= 0.00, "bad"
)